In [1]:
# imports

# show rich formats 
from IPython.display import display, Markdown

# get python to interact with openai
from openai import OpenAI

# use personal openai key
import os
from dotenv import load_dotenv

# check load_dotenv works - should come back 'True'
# load_dotenv()

# make a destination 'resumes' directory for the work

os.makedirs("resumes", exist_ok=True)

# use python to format markdown to html
from markdown import markdown

#### because we're sending this out as a resume, we want it to be in a .pdf format. 
#### that means we'll have to use a library called weasy... 
##### <i><b>< record scratch ></i></b>
#### nope. sorry. 
#### not using weasyprint. lining weasyprint libraries up with each particular python environmet and setting / dedicating kernals still causes nightmares of 'public-speaking-in-underpants' proportions. surely a powerful tool, but it's got the playfulness and ease of use of a rabid porcupine. <i>no grazie</n>.
#### instead, going with pdfkit. working in html, so it'll do the job.
##### * note: pdfkit works using wkhtmltopdf, which <i>in very simple terms</i> converts a webpage to a pdf file. please see [reference.txt](https://github.com/npj210mlk/jobapp_prompter/blob/main/requirements.txt) in the repo for installation instructions.


In [2]:
# import pdfkit

# # test
# pdfkit.from_string("<h1>It should be called 'QueezyPrint,' amirite?</h1>", "output.pdf")

#### ha! apparently, jupyter agrees - came back 'True'

***
#### with the libraries imported we can now break the project down into four (4) steps:
##### 1.) open and read the markdown resume file
#####     * see requirements notes
##### 2.) input the desired job description
##### 3.) template some prompt engineering
##### 4.) convert markdown to html
##### 5.) convert html to pdf
***
*** 
#### Step 1: Open and Read the Markdown Resume
****

In [4]:
# open resume and read it
with open("/Users/nicholasjoseph/Desktop/nj_pm_res.md", "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

# check to see if it worked:
# display(Markdown(resume_string))

# markdown resume displays as expected.

***
#### Step 2: Input the Desired Job Description
***

In [5]:
# job description will be from anywhere, so input is used to pause the program until we find one to copy/paste into this variable 
job_desc_string = input()

 About the job At Intoxalock, a member of the Mindr family of brands, we are dedicated to being a force for good. That's why we provide substance use safety, detection and monitoring products and services that help people live responsibly and keep communities safe.  Always aware. Always Guiding. Never Restricting or judging.  Mindr and our family of brands have helped millions of individuals to live and drive responsibly. Intoxalock, a subsidiary of Mindr, is the country’s largest ignition interlock provider (IIDs) in the United States and the only company working to assist individuals in successfully navigating the often-daunting DUI process. For more than 30 years, Intoxalock has helped millions of people get back on the road safely after a DUI, prevent impaired driving, and save lives.  We are proud to have pioneered alcohol-specific fuel cell technology that sets the industry standard. We provide products and services to consumers and program monitoring authorities to effectively d

***
#### Step 3: Template Some Prompt Engineering
##### - this involves dealing with ChatGPT, particularly the ChatGPT 4o-mini pre-trained transformer.
##### <u>a couple of things about the ChatGPT 4o-mini engine (model)</u>:
#####     a.) it is the smaller, more affordable version of the massive GPT-4o engine available to developers; and
#####     b.) because its focus is on text classificaton / prediction, it is purely a "decoder-only" type transformer
#### The idea is to have ChatGPT create the prompt to respond to the likely Applicant Tracking System being used by the job poster.
##### Reason being, machines talk to machines better than we can. 
***

In [6]:
# making up a random "lambda" function to create the variable "prompt_template"

# the text is what I would be putting into ChatGPT were I to do this one job at a time - prompt engineering

prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts
    - Select only the core competencies most relevant to the job description

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)

3. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

4. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.

5.) **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

In [7]:
# set the prompt variable for our ChatGPT message roles
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 4: Generate the Resume with GPT-4o-mini
##### - Make the api call and tell GPT to write the resume using the prompt from above
***

In [8]:
# set up api client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

# make the call

# set response variable to hold the all info we get back
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    # set our roles up - think of casting a play: "Today, the role of the Expert Resume Writer will be played by the system."
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
    
    # give it some creative license 
    temperature = 0.7
)

# get our response
response_string = response.choices[0].message.content

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-Zc1TwXtDpPr0IS6RN5ix57gZ on tokens per min (TPM): Limit 100000, Used 100000, Requested 3028. Please try again in 21h48m5.76s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

***
#### Step 5: Show Us the Results
***

In [9]:
# separate new resume from improvements AI suggests at 'Additional Suggestions'
response_list = response_string.split("## Additional Suggestions")

In [10]:
display(Markdown(response_list[0]))

# Nicholas Joseph  
**Data Engineer | Technical Account Manager | Product & Data Strategist**  
San Antonio, TX | Remote | Hybrid  
nickpjoseph210@gmail.com | (210) 771-9853  
[LinkedIn](https://www.linkedin.com/in/nickjosephsanantonio/) | [GitHub](https://github.com/npj210mlk)  

---

## Career Summary  
Data Engineer with over 7 years of experience specializing in cloud data architecture and ETL processes. Proven track record of assembling large datasets and designing efficient data pipelines that enhance data accessibility and integrity. Skilled in collaborating with cross-functional teams to translate complex data requirements into actionable insights. Adept at using Azure and Snowflake to deliver robust data solutions that drive business outcomes.

---

## Core Competencies  
- Data Engineering & ETL Pipeline Development  
- Azure Cloud Solutions & Snowflake Expertise  
- SQL & Data Modeling  
- Data Warehouse Architecture & Integration  
- Stakeholder Communication & Presentation  
- Agile Project Delivery  
- Client Relationship Management  
- Data Visualization (Tableau, Looker)  

---

## Professional Experience  

### Cloud Data Engineer / Technical Liaison  
**Spaulding Ridge** | Remote | 07/2022–06/2023  
- Designed and implemented ETL pipelines in Azure, improving data access efficiency by 98.9% for enterprise clients.  
- Developed and maintained data warehouse architectures, ensuring optimal design and accuracy throughout data processing.  
- Collaborated with sales and marketing teams to translate technical deliverables into business-friendly messaging, enhancing client understanding and satisfaction.  

### Junior Data Engineer / Sprint Review Lead  
**Apexon** | Remote | 12/2020–03/2022  
- Led ETL processes and cloud data migration efforts, ensuring compliance with regulatory standards while optimizing data pipelines.  
- Enhanced existing data models to support structured and semi-structured data, contributing to improved data integrity and integrity across projects.  
- Facilitated sprint reviews and demos, effectively communicating project progress and technical details to stakeholders, resulting in a 30% increase in client engagement.  

---

## Certifications & Education  
- **Professional Scrum Master I** – scrum.org (2024)  
- **Project Management Professional (in progress)** – Technical Institute of America (2024)  
- **Data Science Certificate** – Codeup, San Antonio TX (2020)  
- **B.S. Biology** – Concordia University, Austin TX  

---

## Technical Skills  
**Languages & Tools:** Python, SQL, NoSQL, dbt, Matillion, HTML, Go, Spark  
**Cloud Platforms:** Azure, Snowflake, GCP, AWS  
**Data Visualization:** Tableau, Looker  
**Methodologies:** Agile (Scrum/Kanban), SAFe  
**Other:** Data Modeling, Client-Facing Dashboards, User Research, Data Analysis  

---

#

***
#### Step 6: Save the New Resume
##### Great. Hit several snags. You either need WeasyPrint installed in some capacity, or other tools I found (like 'Grip') are difficult to automate and test on MacOS. 
##### Looks like the programming gods have spoken: we're going with WeasyPrint.
##### <center><span style ="color:red"><b><u>To Do This:</u></b></span><center>
##### 1.) completely uninstall existing WeasyPrint's existence from your machine with pip and brew uninstalls;
##### 2.) run a 'brew cleanup' just to wipe out any remnants;
##### 3.) form home directory in Terminal (I'm using Mac), type 'brew install cairo pango gdk-pixbuf libffi' - these are the native Weasyprint dependencies;
##### 4.) set your environment variables in your profile (for me, my ~/.zshrc file) to the following:
###### export DYLD_LIBRARY_PATH=/opt/homebrew/lib:$DYLD_LIBRARY_PATH
###### export LIBRARY_PATH="/opt/homebrew/lib:$LIBRARY_PATH"
###### export PKG_CONFIG_PATH="/opt/homebrew/lib/pkgconfig:/opt/homebrew/share/pkgconfig"
###### export PATH="/opt/homebrew/bin:$PATH"
##### 5.) restart the terminal;
##### 6.) navigate to your project folder;
##### 7.) type 'pip install weasyprint markdown'; and (finally)
##### 8.) open your notebook from your project file with 'jupyter notebook'
***

In [19]:
# Markdown was already imported earlier
# import WeasyPrint and its HTML abilities
from weasyprint import HTML

# save it as PDF
output_pdf_file = "resumes/nick_joseph_tam_resume.pdf"

# convert the Markdown file to HTML
html_content = markdown(response_list[0])

# now take that HTML and convert it into a PDF and save
HTML(string=html_content).write_pdf(output_pdf_file)

In [20]:
# let's save the new Markdown file, too
markdown_output = "resumes/nickjoseph_dataengineer_resume_markdown.md"

with open(markdown_output, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

***
#### Step 7: Display Suggestions for Improvement
##### Because we split "response_string" earlier, it was split at "Additional Suggestions," giving two items "response_list."
##### The first item ([0]) is the resume ChatGPT wrote with our prompt.
##### The second item ([1]) are the suggestions ChatGPT offers to make our resume stronger
***

In [21]:
# show us how to make the resume stronger
display(Markdown(response_list[1]))

  
1. **Skills to Add**:
   - Familiarity with Azure Data Factory or Azure Synapse Analytics to enhance your Azure Cloud experience.
   - Experience with performance tuning for SQL queries to improve data processing speed.

2. **Certifications or Courses**:
   - Consider pursuing an **Azure Data Engineer Associate certification** to solidify your credentials in Azure.
   - Explore advanced SQL courses that focus on performance optimization and complex queries.

3. **Project Ideas**:
   - Develop a personal project that involves building a data pipeline using Azure Data Factory to showcase your ability to handle ETL processes with real-world data.
   - Create a portfolio piece around a data warehouse architecture design that includes documentation on data governance and integrity checks.

By incorporating these suggestions and focusing on your relevant experience, your resume will be more aligned with the Data Engineer role, increasing your chances of landing an interview.

In [14]:
# # markdown is already imported
# from weasyprint import HTML

In [15]:
# from markdown2pdf import convert

# markdown_resume = display(Markdown(response_list[0]))

# convert(markdown_resume, "nj_resume_in_pdf.pdf")